## Statistical method as baseline - CRF

In [26]:
%pip install -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [27]:
import sys
from pathlib import Path

# go up one level from current folder to project root
sys.path.append(str(Path().resolve().parent))

In [28]:
%pip install sklearn-crfsuite rapidfuzz

Note: you may need to restart the kernel to use updated packages.


In [29]:
# parameters

NO_SAMPLES : int   = 50000

# ── Splits ───────────────────────────────────────────────────────────────────
SEED:      int   = 42
N_FOLDS:   int   = 3

In [30]:
from data_preparation.preprocess_data import preprocess_data

df = preprocess_data()[:NO_SAMPLES]

Loading dataset...
Cleaning data...


In [31]:
display(df['target'][0])

{'house_number': '81',
 'street': 'Epovska',
 'city': 'Lubinjë E Epërme',
 'country': 'XK',
 'postal_code': '',
 'state': ''}

In [33]:
from rapidfuzz import fuzz

def align_labels(raw_address: str, parsed: dict) -> list[tuple[str, str]]:
    tokens = raw_address.split()
    labels = ["O"] * len(tokens)

    field_map = {
        "house_number": "NUM",
        "street":       "STR",
        "city":         "CITY",
        "postal_code":  "POST",
        "state":        "STATE",
        "country":      "CTRY",
    }

    # Sort by value length descending — match longer values first
    # avoids short values (like "XK") stealing tokens from longer ones
    sorted_fields = sorted(
        [(f, t) for f, t in field_map.items() if f in parsed],
        key=lambda x: len(str(parsed[x[0]])),
        reverse=True
    )

    for field, tag in sorted_fields:
        value = str(parsed[field])
        value_tokens = value.split()

        # Guard: don't match very short values against wrong tokens
        min_length = 3
        if len(value) < min_length:
            # For short values like "XK", require exact match only
            for i, token in enumerate(tokens):
                if labels[i] == "O" and token.lower() == value.lower():
                    labels[i] = f"B-{tag}"
                    break
            continue

        # Fuzzy sliding window for longer values
        best_score, best_i = 0, -1
        for i in range(len(tokens) - len(value_tokens) + 1):
            # Skip already-labeled tokens
            if any(labels[i+k] != "O" for k in range(len(value_tokens))):
                continue
            window = " ".join(tokens[i:i+len(value_tokens)])
            score = fuzz.ratio(window.lower(), value.lower())
            if score > best_score:
                best_score, best_i = score, i

        if best_score >= 75 and best_i >= 0:
            labels[best_i] = f"B-{tag}"
            for k in range(1, len(value_tokens)):
                labels[best_i + k] = f"I-{tag}"

    return tuple(tokens), tuple(labels)
    # return list(zip(tokens, labels))

In [34]:
def word_features(tokens: list[str], i: int) -> dict:
    word = tokens[i]
    features = {
        "bias":           1.0,
        "word.lower":     word.lower(),
        "word.isupper":   word.isupper(),
        "word.istitle":   word.istitle(),
        "word.isdigit":   word.isdigit(),
        "word.len":       len(word),
        "prefix2":        word[:2].lower(),
        "suffix2":        word[-2:].lower(),
        # Is it a plausible house number?
        "is_num_pattern": bool(re.match(r'^\d+[a-zA-Z]?$', word)),
        # Looks like a postal code?
        "is_postal":      bool(re.match(r'\b[A-Za-z0-9][A-Za-z0-9\- ]{2,9}[A-Za-z0-9]\b', word)),
        # "is_postal":      bool(re.match(r'^\d{5}(-\d{4})?$', word)),
        # 2-letter country code?
        "is_cc":          bool(re.match(r'^[A-Z]{2}$', word)),
        "position":       i,
        "position_rel":   i / max(len(tokens), 1),
    }

    # Context window ±2
    for offset, prefix in [(-2,"m2"), (-1,"m1"), (1,"p1"), (2,"p2")]:
        j = i + offset
        if 0 <= j < len(tokens):
            features[f"{prefix}:word.lower"] = tokens[j].lower()
            features[f"{prefix}:isdigit"]    = tokens[j].isdigit()
            features[f"{prefix}:istitle"]    = tokens[j].istitle()
        else:
            features[f"{prefix}:BOS_EOS"] = True

    return features

In [35]:
import pandas as pd

def prepare_split(split_df: pd.DataFrame):
    tokenized_addresses = []
    aligned_labels = []

    for index, row in split_df.iterrows():
        if row['target'].get('country') == 'Taiwan':
            continue

        raw_address = row['input']
        target_dict = row['target']
        tokens, labels = align_labels(raw_address, target_dict)

        if not tokens:
            continue

        tokenized_addresses.append(tokens)
        aligned_labels.append(labels)

    return tokenized_addresses, aligned_labels

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats
from data_preparation.preprocess_data import make_splits
import regex as re
import sklearn_crfsuite

# ── Step 1: Tune on fold 0 only ───────────────────────────────────────────────
print("Tuning hyperparameters on fold 0...")

first_split = next(make_splits(df))
train_tokens, train_labels = prepare_split(first_split['train'])
val_tokens,   val_labels   = prepare_split(first_split['val'])

X_train = [[word_features(list(t), i) for i in range(len(t))] for t in train_tokens]
X_val   = [[word_features(list(t), i) for i in range(len(t))] for t in val_tokens]
y_train = [list(l) for l in train_labels]
y_val   = [list(l) for l in val_labels]

# Combine train+val for cross-validated search
X_tune = X_train + X_val
y_tune = y_train + y_val

crf = sklearn_crfsuite.CRF(
    algorithm="lbfgs",
    max_iterations=200,
    all_possible_transitions=True,
)

param_distributions = {
    "c1": stats.expon(scale=0.5),   # L1 — controls sparsity
    "c2": stats.expon(scale=0.05),  # L2 — controls smoothness
}

search = RandomizedSearchCV(
    crf,
    param_distributions,
    cv=3,
    n_iter=20,
    scoring="f1_weighted", 
    n_jobs=-1,
    verbose=1,
    random_state=SEED,
)
search.fit(X_tune, y_tune)

best_c1 = search.best_params_["c1"]
best_c2 = search.best_params_["c2"]
print(f"Best params → c1: {best_c1:.4f} | c2: {best_c2:.4f}")

# ── Step 2: Use best params across all folds ──────────────────────────────────
for fold_num, split in enumerate(make_splits(df)):
    print(f"\n--- Fold {fold_num} ---")

    train_tokens, train_labels = prepare_split(split['train'])
    val_tokens,   val_labels   = prepare_split(split['val'])
    test_tokens,  test_labels  = prepare_split(split['test'])

    X_train = [[word_features(list(t), i) for i in range(len(t))] for t in train_tokens]
    print(X_train)
    X_val   = [[word_features(list(t), i) for i in range(len(t))] for t in val_tokens]
    X_test  = [[word_features(list(t), i) for i in range(len(t))] for t in test_tokens]
    y_train = [list(l) for l in train_labels]
    y_val   = [list(l) for l in val_labels]
    y_test  = [list(l) for l in test_labels]

    print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

    crf = sklearn_crfsuite.CRF(
        algorithm="lbfgs",
        c1=best_c1,
        c2=best_c2,
        max_iterations=200,
        all_possible_transitions=True,
    )
    crf.fit(X_train, y_train)

    y_val_pred  = crf.predict(X_val)
    y_test_pred = crf.predict(X_test)
    label_names = [l for l in crf.classes_ if l != "O"]

    print("Val report:")
    print(metrics.flat_classification_report(y_val, y_val_pred, labels=label_names))
    print("Test report:")
    print(metrics.flat_classification_report(y_test, y_test_pred, labels=label_names))

Tuning hyperparameters on fold 0...
Downsampling data from 50000 to 10000 samples...
Fitting 3 folds for each of 20 candidates, totalling 60 fits


/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/model_selection/_validation.py:927: UserWarning: Scoring failed. The score on this train-test partition for these parameters will be set to nan. Details: 
Traceback (most recent call last):
  File "/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/model_selection/_validation.py", line 916, in _score
    scores = scorer(estimator, X_test, y_test, **score_params)
  File "/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/metrics/_scorer.py", line 317, in __call__
    return self._score(partial(_cached_call, None), estimator, X, y_true, **_kwargs)
           ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-a

Best params → c1: 0.2346 | c2: 0.1505
Downsampling data from 50000 to 10000 samples...

--- Fold 0 ---
[{'bias': 1.0, 'word.lower': '99號八樓', 'word.isupper': False, 'word.istitle': False, 'word.isdigit': False, 'word.len': 5, 'prefix2': '99', 'suffix2': '八樓', 'is_num_pattern': False, 'is_postal': False, 'is_cc': False, 'position': 0, 'position_rel': 0.0, 'm2:BOS_EOS': True, 'm1:BOS_EOS': True, 'p1:word.lower': '農權路', 'p1:isdigit': False, 'p1:istitle': False, 'p2:word.lower': 'gönderim', 'p2:isdigit': False, 'p2:istitle': False}, {'bias': 1.0, 'word.lower': '農權路', 'word.isupper': False, 'word.istitle': False, 'word.isdigit': False, 'word.len': 3, 'prefix2': '農權', 'suffix2': '權路', 'is_num_pattern': False, 'is_postal': False, 'is_cc': False, 'position': 1, 'position_rel': 0.125, 'm2:BOS_EOS': True, 'm1:word.lower': '99號八樓', 'm1:isdigit': False, 'm1:istitle': False, 'p1:word.lower': 'gönderim', 'p1:isdigit': False, 'p1:istitle': False, 'p2:word.lower': 'tw', 'p2:isdigit': False, 'p2:istitle

# Libpostal

In [4]:
%pip install postal

Note: you may need to restart the kernel to use updated packages.


In [ ]:
from postal.parser import parse_address

# ── Mapowanie libpostal → nasze tagi ─────────────────────────────────────────
LIBPOSTAL_MAP = {
    "house":             "NUM",
    "house_number":      "NUM",
    "road":              "STR",
    "city":              "CITY",
    "postcode":          "POST",
    "state":             "STATE",
    "country":           "CTRY",
    "country_region":    "CTRY",
}

def libpostal_predict(raw_address: str, tokens: tuple) -> list[str]:
    parsed = parse_address(raw_address)

    value_to_tag = {}
    for value, component in parsed:
        tag = LIBPOSTAL_MAP.get(component)
        if tag:
            # multi-tokenowe wartości np. "Lubinjë E Epërme"
            for sub_token in value.lower().split():
                value_to_tag[sub_token] = tag

    labels = ["O"] * len(tokens)
    prev_tag = None

    for i, token in enumerate(tokens):
        tag = value_to_tag.get(token.lower())
        if tag:
            if tag == prev_tag:
                labels[i] = f"I-{tag}"   # kontynuacja spanu
            else:
                labels[i] = f"B-{tag}"   # nowy span
            prev_tag = tag
        else:
            prev_tag = None

    return labels


# ── Ewaluacja na zbiorze testowym ─────────────────────────────────────────────
from sklearn_crfsuite import metrics
from sklearn.metrics import classification_report

def evaluate_libpostal(test_df: pd.DataFrame):
    y_true = []
    y_pred = []

    for _, row in test_df.iterrows():
        if row['target'].get('country') == 'Taiwan':
            continue

        raw_address = row['input']
        target_dict = row['target']

        tokens, true_labels = align_labels(raw_address, target_dict)

        if not tokens:
            continue

        pred_labels = libpostal_predict(raw_address, tokens)

        y_true.append(list(true_labels))
        y_pred.append(pred_labels)

    label_names = list({l for seq in y_true for l in seq if l != "O"})
    print(metrics.flat_classification_report(y_true, y_pred, labels=sorted(label_names)))

    return y_true, y_pred

In [96]:
for fold_num, split in enumerate(make_splits(df)):
    print(f"\n═══════════ Fold {fold_num} ═══════════")

    train_tokens, train_labels = prepare_split(split['train'])
    test_tokens,  test_labels  = prepare_split(split['test'])

    # ── CRF ──────────────────────────────────────────────────────────────────
    X_train = [[word_features(list(t), i) for i in range(len(t))] for t in train_tokens]
    X_test  = [[word_features(list(t), i) for i in range(len(t))] for t in test_tokens]
    y_train = [list(l) for l in train_labels]
    y_test  = [list(l) for l in test_labels]

    crf = sklearn_crfsuite.CRF(
        algorithm="lbfgs",
        c1=best_c1,
        c2=best_c2,
        max_iterations=200,
        all_possible_transitions=True,
    )
    crf.fit(X_train, y_train)

    y_crf_pred  = crf.predict(X_test)
    label_names = sorted([l for l in crf.classes_ if l != "O"])

    print("\n── CRF ──")
    print(metrics.flat_classification_report(y_test, y_crf_pred, labels=label_names))

    # ── libpostal ─────────────────────────────────────────────────────────────
    print("\n── libpostal ──")
    evaluate_libpostal(split['test'])  # ← to wszystko, przekazujesz split['test'] bezpośrednio

Downsampling data from 50000 to 10000 samples...

═══════════ Fold 0 ═══════════

── CRF ──
              precision    recall  f1-score   support

      B-CITY       0.86      0.87      0.87      1442
      B-CTRY       0.99      1.00      1.00      1270
       B-NUM       0.95      0.95      0.95      1446
      B-POST       0.86      0.87      0.87       224
     B-STATE       0.97      0.97      0.97      1222
       B-STR       0.82      0.85      0.83      1253
      I-CITY       0.74      0.57      0.64        49
     I-STATE       0.89      0.84      0.86        19
       I-STR       0.74      0.84      0.79       568

   micro avg       0.90      0.92      0.91      7493
   macro avg       0.87      0.86      0.86      7493
weighted avg       0.90      0.92      0.91      7493


── libpostal ──


/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precisi

              precision    recall  f1-score   support

      B-CITY       0.54      0.35      0.43      1442
      B-CTRY       0.94      0.72      0.81      1270
       B-NUM       0.81      0.72      0.76      1446
      B-POST       0.62      0.63      0.62       224
     B-STATE       0.90      0.37      0.53      1222
       B-STR       0.59      0.25      0.36      1253
      I-CITY       0.00      0.00      0.00        49
     I-STATE       0.00      0.00      0.00        19
       I-STR       0.00      0.00      0.00       568

   micro avg       0.75      0.45      0.56      7493
   macro avg       0.49      0.34      0.39      7493
weighted avg       0.68      0.45      0.53      7493


═══════════ Fold 1 ═══════════

── CRF ──
              precision    recall  f1-score   support

      B-CITY       0.85      0.86      0.86      1422
      B-CTRY       0.99      1.00      0.99      1276
       B-NUM       0.95      0.95      0.95      1449
      B-POST       0.86      0.83  

/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precisi


── CRF ──
              precision    recall  f1-score   support

      B-CITY       0.85      0.87      0.86      1436
      B-CTRY       0.99      1.00      0.99      1279
       B-NUM       0.95      0.95      0.95      1445
      B-POST       0.85      0.87      0.86       237
     B-STATE       0.97      0.97      0.97      1231
       B-STR       0.84      0.85      0.84      1264
      I-CITY       0.77      0.57      0.65        58
     I-STATE       0.88      0.75      0.81        20
       I-STR       0.79      0.79      0.79       614

   micro avg       0.90      0.91      0.91      7584
   macro avg       0.88      0.85      0.86      7584
weighted avg       0.90      0.91      0.91      7584


── libpostal ──
              precision    recall  f1-score   support

      B-CITY       0.49      0.33      0.39      1436
      B-CTRY       0.94      0.73      0.82      1279
       B-NUM       0.80      0.72      0.76      1445
      B-POST       0.62      0.65      0.64       

/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/weronikadomczewska/Documents/magisterskie/semestr3/NLP/NLP-address-data-parsing/.venv/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precisi

In [97]:
def bio_to_dict(tokens: tuple, labels: list[str]) -> dict:
    """Konwertuje tokeny + BIO labels → słownik adresu."""
    TAG_TO_FIELD = {
        "NUM":   "house_number",
        "STR":   "street",
        "CITY":  "city",
        "POST":  "postal_code",
        "STATE": "state",
        "CTRY":  "country",
    }

    result = {}
    current_tag, current_tokens = None, []

    for token, label in zip(tokens, labels):
        if label.startswith("B-"):
            if current_tag:
                field = TAG_TO_FIELD.get(current_tag)
                if field:
                    result[field] = " ".join(current_tokens)
            current_tag = label[2:]
            current_tokens = [token]
        elif label.startswith("I-") and label[2:] == current_tag:
            current_tokens.append(token)
        else:
            if current_tag:
                field = TAG_TO_FIELD.get(current_tag)
                if field:
                    result[field] = " ".join(current_tokens)
            current_tag, current_tokens = None, []

    if current_tag:
        field = TAG_TO_FIELD.get(current_tag)
        if field:
            result[field] = " ".join(current_tokens)

    return result

In [99]:
from data_preparation.preprocess_data import evaluate_predictions

for fold_num, split in enumerate(make_splits(df)):
    print(f"\n═══════════ Fold {fold_num} ═══════════")

    train_tokens, train_labels = prepare_split(split['train'])
    test_tokens,  test_labels  = prepare_split(split['test'])

    # ── Trenowanie CRF ───────────────────────────────────────────────────────
    X_train = [[word_features(list(t), i) for i in range(len(t))] for t in train_tokens]
    X_test  = [[word_features(list(t), i) for i in range(len(t))] for t in test_tokens]
    y_train = [list(l) for l in train_labels]

    crf = sklearn_crfsuite.CRF(
        algorithm="lbfgs", c1=best_c1, c2=best_c2,
        max_iterations=200, all_possible_transitions=True,
    )
    crf.fit(X_train, y_train)

    # ── CRF → słowniki ───────────────────────────────────────────────────────
    y_crf_pred = crf.predict(X_test)

    crf_pred_dicts  = [bio_to_dict(t, l) for t, l in zip(test_tokens, y_crf_pred)]
    gold_dicts      = list(split['test']['target'])  # już są słownikami

    # ── evaluate_predictions ─────────────────────────────────────────────────
    print("\n── CRF ──")
    crf_results = evaluate_predictions(gold_dicts, crf_pred_dicts)
    print(f"Exact match:          {crf_results['exact_match']:.3f}")
    print(f"Overall item acc:     {crf_results['overall_item_accuracy']:.3f}")
    print(f"Per field accuracy:")
    for field, acc in crf_results['per_field_accuracy'].items():
        print(f"  {field:<15} {acc:.3f}")

    # ── libpostal → słowniki ─────────────────────────────────────────────────
    print("\n── libpostal ──")
    libpostal_pred_dicts = []
    for _, row in split['test'].iterrows():
        if row['target'].get('country') == 'Taiwan':
            continue
        tokens, _ = align_labels(row['input'], row['target'])
        pred_labels = libpostal_predict(row['input'], tokens)
        libpostal_pred_dicts.append(bio_to_dict(tokens, pred_labels))

    libpostal_results = evaluate_predictions(gold_dicts, libpostal_pred_dicts)
    print(f"Exact match:          {libpostal_results['exact_match']:.3f}")
    print(f"Overall item acc:     {libpostal_results['overall_item_accuracy']:.3f}")
    print(f"Per field accuracy:")
    for field, acc in libpostal_results['per_field_accuracy'].items():
        print(f"  {field:<15} {acc:.3f}")

Downsampling data from 50000 to 10000 samples...

═══════════ Fold 0 ═══════════

── CRF ──
Exact match:          0.295
Overall item acc:     0.791
Per field accuracy:
  house_number    0.853
  street          0.525
  city            0.702
  country         0.848
  postal_code     0.961
  state           0.859

── libpostal ──
Exact match:          0.017
Overall item acc:     0.496
Per field accuracy:
  house_number    0.621
  street          0.140
  city            0.297
  country         0.607
  postal_code     0.899
  state           0.410

═══════════ Fold 1 ═══════════

── CRF ──
Exact match:          0.285
Overall item acc:     0.785
Per field accuracy:
  house_number    0.853
  street          0.515
  city            0.681
  country         0.853
  postal_code     0.953
  state           0.855

── libpostal ──
Exact match:          0.018
Overall item acc:     0.475
Per field accuracy:
  house_number    0.611
  street          0.141
  city            0.263
  country         0.580